# Painel de Análise de Vendas com Visualização de Dados

Notebook completo das etapas de preparação, análise exploratória, visualização e apresentação.  
O código cria um conjunto de dados fictício caso `vendas.csv` não esteja disponível. Se o arquivo existir, ele será utilizado.

**Colunas esperadas:** `data`, `loja`, `produto`, `quantidade`, `preco_unitario`, `regiao` e `categoria`.


## Etapa 1 — Preparação e importação de dados

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="deep")

In [ ]:
# Carrega vendas.csv; caso não exista, cria dados fictícios reprodutíveis.
arquivo = "vendas.csv"

if os.path.exists(arquivo):
    df = pd.read_csv(arquivo)
else:
    rng = np.random.default_rng(42)
    n = 500
    produtos = ["Notebook", "Mouse", "Teclado", "Monitor", "Headset",
                "Webcam", "Impressora", "Cadeira"]
    lojas = ["Loja Centro", "Loja Norte", "Loja Sul", "Loja Oeste"]
    regioes = {"Loja Centro": "Sudeste", "Loja Norte": "Norte",
               "Loja Sul": "Sul", "Loja Oeste": "Centro-Oeste"}
    categorias = {"Notebook": "Informática", "Mouse": "Acessórios",
                  "Teclado": "Acessórios", "Monitor": "Informática",
                  "Headset": "Acessórios", "Webcam": "Acessórios",
                  "Impressora": "Informática", "Cadeira": "Móveis"}
    precos = {"Notebook": 3500, "Mouse": 80, "Teclado": 150,
              "Monitor": 1200, "Headset": 220, "Webcam": 180,
              "Impressora": 900, "Cadeira": 700}
    prod = rng.choice(produtos, n)
    loja = rng.choice(lojas, n)
    df = pd.DataFrame({
        "data": rng.choice(pd.date_range("2025-01-01", "2025-12-31", freq="D"), n),
        "loja": loja,
        "produto": prod,
        "quantidade": rng.integers(1, 8, n),
        "preco_unitario": [precos[p] * rng.uniform(0.9, 1.1) for p in prod],
        "regiao": [regioes[l] for l in loja],
        "categoria": [categorias[p] for p in prod]
    })
    df["preco_unitario"] = df["preco_unitario"].round(2)
    df.to_csv(arquivo, index=False)
    print("Arquivo fictício vendas.csv criado.")

df.head()

In [ ]:
# Conversão da data e criação do faturamento
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["faturamento"] = df["quantidade"] * df["preco_unitario"]

print("Informações do conjunto:")
df.info()

In [ ]:
# Integridade: tipos, valores ausentes e resumo estatístico
display(df.isnull().sum().to_frame("valores_ausentes"))
display(df.describe(include="all").T)

In [ ]:
# Trata valores ausentes numéricos pela média da respectiva coluna
for coluna in ["preco_unitario", "quantidade"]:
    df[coluna] = df[coluna].fillna(df[coluna].mean())

# Recalcula o faturamento após o tratamento
df["faturamento"] = df["quantidade"] * df["preco_unitario"]

## Etapa 2 — Análise Exploratória e Estatísticas

In [ ]:
# Faturamento total por loja, do maior para o menor
faturamento_por_loja = (
    df.groupby("loja")["faturamento"].sum().sort_values(ascending=False)
)
display(faturamento_por_loja.to_frame("faturamento_total"))

In [ ]:
# Ticket médio por loja: faturamento total / número de vendas
ticket_medio = (
    df.groupby("loja")["faturamento"].sum() /
    df.groupby("loja").size()
).sort_values(ascending=False)
display(ticket_medio.to_frame("ticket_medio"))

In [ ]:
# Cinco produtos mais vendidos em quantidade
top5_produtos = df.groupby("produto")["quantidade"].sum().nlargest(5)
display(top5_produtos.to_frame("quantidade_vendida"))

In [ ]:
# Mês com maior e menor faturamento
df["mes"] = df["data"].dt.to_period("M")
faturamento_mensal = df.groupby("mes")["faturamento"].sum().sort_index()
mes_maior = faturamento_mensal.idxmax()
mes_menor = faturamento_mensal.idxmin()
print(f"Maior faturamento: {mes_maior.strftime('%m/%Y')} — R$ {faturamento_mensal.max():,.2f}")
print(f"Menor faturamento: {mes_menor.strftime('%m/%Y')} — R$ {faturamento_mensal.min():,.2f}")

In [ ]:
# Coluna numérica do mês (1 a 12)
df["mes_numero"] = df["data"].dt.month
df[["data", "mes", "mes_numero"]].head()

## Etapa 3 — Visualizações com matplotlib e seaborn

In [ ]:
# 11. Gráfico de barras: faturamento por loja
fig, ax = plt.subplots(figsize=(9, 5))
faturamento_por_loja.plot(kind="bar", ax=ax)
ax.set(title="Faturamento total por loja", xlabel="Loja", ylabel="Faturamento (R$)")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
os.makedirs("graficos", exist_ok=True)
fig.savefig("graficos/faturamento_por_loja.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 12. Gráfico de linhas: evolução mensal do faturamento
mensal = df.groupby(df["data"].dt.to_period("M"))["faturamento"].sum()
mensal.index = mensal.index.astype(str)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(mensal.index, mensal.values, marker="o", label="Faturamento")
for x, y in zip(mensal.index, mensal.values):
    ax.annotate(f"R$ {y:,.0f}", (x, y), textcoords="offset points",
                xytext=(0, 7), ha="center", fontsize=8)
ax.set(title="Evolução mensal do faturamento", xlabel="Mês/Ano", ylabel="Faturamento (R$)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
fig.savefig("graficos/evolucao_mensal.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 13. Gráfico de pizza: faturamento por região
faturamento_regiao = df.groupby("regiao")["faturamento"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
ax.pie(faturamento_regiao, labels=faturamento_regiao.index, autopct="%1.1f%%",
       startangle=90, wedgeprops={"edgecolor": "white"})
ax.set_title("Distribuição do faturamento por região")
ax.axis("equal")
plt.tight_layout()
fig.savefig("graficos/faturamento_por_regiao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 14. Dispersão: quantidade x faturamento
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x="quantidade", y="faturamento", hue="categoria",
                alpha=0.7, ax=ax)
ax.set(title="Relação entre quantidade vendida e faturamento",
       xlabel="Quantidade", ylabel="Faturamento (R$)")
plt.tight_layout()
fig.savefig("graficos/quantidade_faturamento.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 15. Boxplot: distribuição dos preços por categoria
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="categoria", y="preco_unitario", hue="categoria", legend=False, ax=ax)
ax.set(title="Distribuição dos preços por categoria",
       xlabel="Categoria", ylabel="Preço unitário (R$)")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig("graficos/precos_por_categoria.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 16. Heatmap: correlação entre variáveis numéricas
correlacao = df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(correlacao, annot=True, cmap="coolwarm", fmt=".2f", ax=ax)
ax.set_title("Correlação entre variáveis numéricas")
plt.tight_layout()
fig.savefig("graficos/matriz_correlacao.png", dpi=150, bbox_inches="tight")
plt.show()

## Etapa 4 — Estilização e apresentação

In [ ]:
# 17. Tema global e paleta de cores
sns.set_style("whitegrid")
sns.set_palette("deep")

In [ ]:
# 18. Os gráficos acima possuem títulos e rótulos descritivos.
# 19. Comparação lado a lado: faturamento por loja e por região.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
faturamento_por_loja.plot(kind="bar", ax=axes[0])
axes[0].set(title="Faturamento por loja", xlabel="Loja", ylabel="Faturamento (R$)")
axes[0].tick_params(axis="x", rotation=25)
faturamento_regiao.plot(kind="bar", ax=axes[1])
axes[1].set(title="Faturamento por região", xlabel="Região", ylabel="Faturamento (R$)")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig("graficos/comparacao_loja_regiao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 20. Exportação dos gráficos e do resumo analítico
resumo_analitico = pd.DataFrame({
    "indicador": [
        "Faturamento total",
        "Loja com maior faturamento",
        "Maior faturamento mensal",
        "Mês de maior faturamento",
        "Menor faturamento mensal",
        "Mês de menor faturamento"
    ],
    "valor": [
        df["faturamento"].sum(),
        faturamento_por_loja.idxmax(),
        faturamento_mensal.max(),
        mes_maior.strftime("%m/%Y"),
        faturamento_mensal.min(),
        mes_menor.strftime("%m/%Y")
    ]
})
resumo_analitico.to_csv("resumo_analitico.csv", index=False)
display(resumo_analitico)
print("Gráficos salvos na pasta 'graficos/' e resumo em resumo_analitico.csv.")

## Conclusão

O notebook prepara os dados, verifica e trata valores ausentes, calcula indicadores de vendas e gera gráficos para comparar lojas, regiões, produtos e meses. Os resultados numéricos dependem do arquivo `vendas.csv`; sem ele, são utilizados dados fictícios para demonstrar o funcionamento.